# PII Masking Pipeline Demonstration

This notebook demonstrates the end-to-end modular PII Masking Pipeline. We will load the dataset, assemble and run our pipeline step-by-step, run evaluations after each step to track performance improvements, and visualize the comparative metrics.

### Pipeline Stages
1. **Baseline spaCy NLP Engine** - Basic tokenization and standard entities.
2. **Custom Pattern Recognizers** - Regular expressions optimized for phone numbers, tax IDs, and location keywords.
3. **Contextual Deep Learning Model** - Contextual transformer model (e.g. Vietnamese ELECTRA) to detect names, organizations, and locations.

In [ ]:
# Setup Python path to find the src directory relative to this notebook
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Quick Example: Put in custom text and see the pipeline's prediction
from src.pipeline.Recognizers.SpacyRecognizer import SpacyRecognizer
from src.pipeline.Recognizers.CustomPatternRecognizer import CustomPatternRecognizer
from src.pipeline.Recognizers.TransformersRecognizer import TransformersRecognizer
from src.pipeline.BasePipeline import PIIPipeline
from presidio_anonymizer import AnonymizerEngine

print("Initializing the full hybrid pipeline...")

# 1. Initialize recognizers
spacy_rec = SpacyRecognizer(model_name="xx_ent_wiki_sm", lang_code="vi")
custom_patterns = CustomPatternRecognizer()
transformers_rec = TransformersRecognizer(
    model_id="NlpHUST/ner-vietnamese-electra-base",
    label_mapping={"PER": "PERSON", "LOC": "LOCATION", "ORG": "ORGANIZATION", "MISC": "MISC"},
    lang_code="vi"
)

# 2. Assemble complete pipeline
demo_pipeline = PIIPipeline(
    spacy_recognizer=spacy_rec,
    recognizers=[custom_patterns, transformers_rec]
)
demo_pipeline.load_model()
print("Pipeline loaded successfully.")

# --- CUSTOM TEXT INPUT ---
# You can modify this text to test any PII identification and masking
user_text = "Xin chào, tôi là Nguyễn Văn A. Số điện thoại của tôi là 0912345678. Tôi sống tại Hà Nội và làm việc tại Công ty ABC."

# Run prediction
results = demo_pipeline.predict(user_text, language="vi")

# Apply anonymization
anonymizer = AnonymizerEngine()
anonymized_result = anonymizer.anonymize(text=user_text, analyzer_results=results)

print("\n--- Results ---")
print("Original Text:", user_text)
print("\nDetected Entities:")
for res in results:
    entity_text = user_text[res.start:res.end]
    print(f"- [{res.entity_type}] (Score: {res.score:.2f}) -> '{entity_text}'")

print("\nAnonymized Text Output:")
print(anonymized_result.text)

## Dataset Selection and Setup

We load the PII dataset `NAMANDREWLV/pii-masking-95k-preencoded` using Hugging Face datasets. You can select the specific split to test: `train`, `val`, `test` or `all` to combine them.

In [ ]:
import pandas as pd
from datasets import load_dataset

# --- CONFIGURATION ---
# Choose dataset split to test: 'train', 'val', 'test', or 'all'
split_choice = "test"

# Set the number of rows to sample to keep evaluation fast
sample_size = 5000

print(f"Loading dataset: NAMANDREWLV/pii-masking-95k-preencoded")
dataset = load_dataset("NAMANDREWLV/pii-masking-95k-preencoded")

available_splits = list(dataset.keys())
print("Available splits in dataset:", available_splits)

# Map splits
selected_splits = []
if split_choice == "all":
    selected_splits = available_splits
else:
    mapped_split = split_choice
    if split_choice == "val" and "validation" in dataset:
        mapped_split = "validation"
    
    if mapped_split in dataset:
        selected_splits = [mapped_split]
    else:
        print(f"Split '{split_choice}' not found, falling back to 'train'.")
        selected_splits = ["train"]

# Load and sample data
dfs = []
for split in selected_splits:
    split_df = dataset[split].to_pandas()
    if len(split_df) > sample_size:
        split_df = split_df.sample(n=sample_size, random_state=42)
    split_df["split"] = split
    dfs.append(split_df)

df_eval = pd.concat(dfs, ignore_index=True)

# Ensure compatibility with our evaluator keys
if "text" in df_eval.columns and "source_text" not in df_eval.columns:
    df_eval["source_text"] = df_eval["text"]

print(f"Loaded and sampled {len(df_eval)} rows across splits: {selected_splits}")

## Initialize Evaluator

We setup the evaluator using standard mappings and prepare a list to hold metrics of each pipeline stage.

In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.pipeline.Evaluator import PIIEvaluator
from src.pipeline.Utils import with_prefix

evaluator = PIIEvaluator()
all_step_results = []
all_per_entity_results = []

### Step 1: Baseline spaCy Recognizer

Evaluate the core spaCy engine baseline setup without custom matchers or deep learning.

In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from src.pipeline.Recognizers.SpacyRecognizer import SpacyRecognizer
from src.pipeline.BasePipeline import PIIPipeline

print("Starting Step 1 Evaluation...")
spacy_rec = SpacyRecognizer(model_name="xx_ent_wiki_sm", lang_code="vi")
pipeline_step1 = PIIPipeline(spacy_recognizer=spacy_rec)
pipeline_step1.load_model()

step1_overall, step1_per_entity = evaluator.evaluate_presidio(
    df_eval=df_eval,
    analyzer=pipeline_step1,
    language="vi",
    use_type_mapping=True,
    return_per_entity=True
)

prefixed_step1 = with_prefix("mapped_types", step1_overall)
prefixed_step1["step_name"] = "Step 1: Spacy Base"
all_step_results.append(prefixed_step1)

# Format per-entity results into a DataFrame
step1_per_entity_df = pd.DataFrame.from_dict(step1_per_entity, orient="index").reset_index().rename(columns={"index": "entity_type"})
step1_per_entity_df["step_name"] = "Step 1: Spacy Base"
all_per_entity_results.append(step1_per_entity_df)

print("\nStep 1 Baseline Metrics:")
for k, v in step1_overall.items():
    print(f"- {k}: {v}")

print("\nStep 1 Per-Entity Metrics:")
for ent, metrics in step1_per_entity.items():
    print(f"- {ent}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1']:.4f}")

### Step 2: Custom Pattern Recognizers

We inject our custom regex pattern recognizers to register rules for phone numbers, IDs, and specific address keywords.

In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from src.pipeline.Recognizers.SpacyRecognizer import SpacyRecognizer
from src.pipeline.Recognizers.CustomPatternRecognizer import CustomPatternRecognizer
from src.pipeline.BasePipeline import PIIPipeline

print("Starting Step 2 Evaluation...")
spacy_rec = SpacyRecognizer(model_name="xx_ent_wiki_sm", lang_code="vi")
pipeline_step2 = PIIPipeline(spacy_recognizer=spacy_rec)
custom_patterns = CustomPatternRecognizer()
pipeline_step2.add_recognizer(custom_patterns)
pipeline_step2.load_model()

step2_overall, step2_per_entity = evaluator.evaluate_presidio(
    df_eval=df_eval,
    analyzer=pipeline_step2,
    language="vi",
    use_type_mapping=True,
    return_per_entity=True
)

prefixed_step2 = with_prefix("mapped_types", step2_overall)
prefixed_step2["step_name"] = "Step 2: Spacy + Regex"
all_step_results.append(prefixed_step2)

# Format per-entity results into a DataFrame
step2_per_entity_df = pd.DataFrame.from_dict(step2_per_entity, orient="index").reset_index().rename(columns={"index": "entity_type"})
step2_per_entity_df["step_name"] = "Step 2: Spacy + Regex"
all_per_entity_results.append(step2_per_entity_df)

print("\nStep 2 (Spacy + Regex) Metrics:")
for k, v in step2_overall.items():
    print(f"- {k}: {v}")

print("\nStep 2 Per-Entity Metrics:")
for ent, metrics in step2_per_entity.items():
    print(f"- {ent}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1']:.4f}")

### Step 3: Transformers Recognizer

We add the deep learning Vietnamese ELECTRA model (`NlpHUST/ner-vietnamese-electra-base`) to identify names, locations, and organizations under deep contextual processing.

In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from src.pipeline.Recognizers.SpacyRecognizer import SpacyRecognizer
from src.pipeline.Recognizers.CustomPatternRecognizer import CustomPatternRecognizer
from src.pipeline.Recognizers.TransformersRecognizer import TransformersRecognizer
from src.pipeline.BasePipeline import PIIPipeline

print("Starting Step 3 Evaluation...")
spacy_rec = SpacyRecognizer(model_name="xx_ent_wiki_sm", lang_code="vi")
pipeline_step3 = PIIPipeline(spacy_recognizer=spacy_rec)
custom_patterns = CustomPatternRecognizer()
pipeline_step3.add_recognizer(custom_patterns)

transformers_rec = TransformersRecognizer(
    model_id="NlpHUST/ner-vietnamese-electra-base",
    label_mapping={"PER": "PERSON", "LOC": "LOCATION", "ORG": "ORGANIZATION", "MISC": "MISC"},
    lang_code="vi"
)
pipeline_step3.add_recognizer(transformers_rec)
pipeline_step3.load_model()

step3_overall, step3_per_entity = evaluator.evaluate_presidio(
    df_eval=df_eval,
    analyzer=pipeline_step3,
    language="vi",
    use_type_mapping=True,
    return_per_entity=True
)

prefixed_step3 = with_prefix("mapped_types", step3_overall)
prefixed_step3["step_name"] = "Step 3: Full Hybrid"
all_step_results.append(prefixed_step3)

# Format per-entity results into a DataFrame
step3_per_entity_df = pd.DataFrame.from_dict(step3_per_entity, orient="index").reset_index().rename(columns={"index": "entity_type"})
step3_per_entity_df["step_name"] = "Step 3: Full Hybrid"
all_per_entity_results.append(step3_per_entity_df)

print("\nStep 3 (Full Hybrid) Metrics:")
for k, v in step3_overall.items():
    print(f"- {k}: {v}")

print("\nStep 3 Per-Entity Metrics:")
for ent, metrics in step3_per_entity.items():
    print(f"- {ent}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1']:.4f}")

## Results Comparison and Plotting

We summarize all metrics across the three stages of the pipeline evolution and visualize the improvement using our plotting helper.

In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from src.pipeline.Utils import display_evaluation_results, plot_step_progress

# Create final comparison dataframes
overall_df = pd.DataFrame(all_step_results)
per_entity_df = pd.concat(all_per_entity_results, ignore_index=True) if all_per_entity_results else None

# Display styled table results (both overall and per-entity)
display_evaluation_results(overall_df, per_entity_df)

# Render improvement metrics plot
plot_step_progress(overall_df)